### Cleaning of the EGFR dataset extracted from Excape db. 


In [ ]:
# importing libraries

import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools
import useful_rdkit_utils as uru
from collections import Counter

In [ ]:
ro5_calc = uru.Ro5Calculator()

In [ ]:
# provide path
path = './raw/EGFR_actives_from_Excape.csv'
df = pd.read_csv(path)

In [ ]:
df.info()

In [ ]:
df = df[['SMILES','pXC50']]

In [ ]:
#rename columns
df.columns = ['smiles','value']

In [ ]:
#check for null values
df.isnull().sum()

In [ ]:
# drop all rows with null values
df = df.dropna(axis = 'rows')

In [ ]:
# generate canonical SMILES to double-check for duplicates

canonical_smiles = []
for s in df.smiles:
    try:
        c = Chem.MolToSmiles(Chem.MolFromSmiles(s))
        canonical_smiles.append(c)
    except:
        canonical_smiles.append(None)


In [ ]:
df['canonical_smiles'] = canonical_smiles

In [ ]:
# Perform the necessary steps before checking

df_nuniques = df.groupby('canonical_smiles').agg({'smiles' : 'nunique','value': 'nunique'}).reset_index(drop=False)
df_first = df.groupby('canonical_smiles').agg({'smiles' : 'first','value': 'unique' }).reset_index(drop=False)


In [ ]:
df_w_info = df_nuniques.merge(df_first[['canonical_smiles','value']], on='canonical_smiles').rename(columns={'value_x' : 'nb_values', 'value_y' : 'values'})

In [ ]:
df_w_info.info()

In [ ]:
df_w_info.head()

In [ ]:
#df_w_info['ROMol'] = [ Chem.MolFromSmiles(s) for s in df_w_info.canonical_smiles]

In [ ]:
df_w_info[ro5_calc.names] = df_w_info.canonical_smiles.apply(ro5_calc.calc_smiles).to_list()


In [ ]:
df_w_info.describe()

The distributions indicate that 
- some compounds are associated with more than 1 activity value
- some compounds have extreme values of Ro5 descriptors.

We select only compounds with one unique pIC50 value and use relaxed Ro5 criteria to clean the dataset before use

In [ ]:
criteria_query = """nb_values < 2  and smiles < 2 and MolWt <= 600 and LogP <= 6 and HBD <= 6 and HBA <= 10 and TPSA <= 150"""
df_remaining = df_w_info.query(criteria_query)

In [ ]:
df_remaining.drop(columns=['smiles'], inplace=True)


In [ ]:
df_remaining.rename(columns={'canonical_smiles' : 'smiles'}, inplace=True)

In [ ]:
df_remaining.info()

In [ ]:
activity_value = [ v[0] for v in df_remaining['values']]

df_remaining['value'] = activity_value

In [ ]:
df_remaining[['smiles', 'value']].to_csv('./egfr_cleaned.csv', index=None)